In [2]:
import pandas as pd
import numpy as np
import os
from PIL import Image

print("✅ Library successfully imported!")

✅ Library successfully imported!


In [3]:
# Load the filtered dataset
df = pd.read_csv('../labels_filtered.csv')

# Create image path column
image_folder = '../Images'
df['image_path'] = df['Image Name'].apply(
    lambda x: os.path.join(image_folder, x)
)

print("Dataset loaded:")
print(df.head())
print(f"\nTotal data: {len(df)}")

Dataset loaded:
  Image Name  Patient Label  Quality Score  label_numeric         image_path
0    0_0.jpg        0  GON+           6.18              1  ../Images/0_0.jpg
1    1_0.jpg        1  GON+           5.31              1  ../Images/1_0.jpg
2    1_1.jpg        1  GON+           4.37              1  ../Images/1_1.jpg
3    2_1.jpg        2  GON+           4.24              1  ../Images/2_1.jpg
4    3_0.jpg        3  GON+           5.95              1  ../Images/3_0.jpg

Total data: 701


In [4]:
# Create a new folder for the resized images.
resized_folder = '../Images_resized'

if not os.path.exists(resized_folder):
    os.makedirs(resized_folder)
    print("✅ Folder Images_resized created!")

# Resize all images
total = len(df)
for i, (index, row) in enumerate(df.iterrows()):
    img_path = os.path.join(image_folder, row['Image Name'])
    img = Image.open(img_path)
    img = img.resize((224, 224))
    save_path = os.path.join(resized_folder, row['Image Name'])
    img.save(save_path)
    
    # Progress every 100 images
    if (i + 1) % 100 == 0:
        print(f"Progress: {i+1}/{total} images completed...")

print(f"\n✅ All {total} images successfully resized to 224x224!")

Progress: 100/701 images completed...
Progress: 200/701 images completed...
Progress: 300/701 images completed...
Progress: 400/701 images completed...
Progress: 500/701 images completed...
Progress: 600/701 images completed...
Progress: 700/701 images completed...

✅ All 701 images successfully resized to 224x224!


In [5]:
# Verify image size after resizing
sample_images = os.listdir(resized_folder)[:5]

print("Check size of 5 images after resizing:")
for img_name in sample_images:
    img_path = os.path.join(resized_folder, img_name)
    with Image.open(img_path) as img:
        print(f"✅ {img_name} → {img.size}")

print(f"\nTotal images in resized folder: {len(os.listdir(resized_folder))}")

Check size of 5 images after resizing:
✅ 94_1.jpg → (224, 224)
✅ 194_0.jpg → (224, 224)
✅ 164_1.jpg → (224, 224)
✅ 166_0.jpg → (224, 224)
✅ 200_0.jpg → (224, 224)

Total images in resized folder: 701


In [6]:
# Convert image to numpy array + normalization
images = []
labels = []

total = len(df)
for i, (index, row) in enumerate(df.iterrows()):
    img_path = os.path.join(resized_folder, row['Image Name'])
    img = Image.open(img_path).convert('RGB')
    img_array = np.array(img) / 255.0
    images.append(img_array)
    labels.append(row['label_numeric'])
    
    # Progress every 100 images
    if (i + 1) % 100 == 0:
        print(f"Progress: {i+1}/{total} images processed...")

images = np.array(images)
labels = np.array(labels)

print(f"\nImages Shape: {images.shape}")
print(f"Labels Shape: {labels.shape}")
print(f"Pixel Value Min: {images.min():.2f}")
print(f"Pixel Value Max: {images.max():.2f}")
print("\n✅ All images successfully converted!")

Progress: 100/701 images processed...
Progress: 200/701 images processed...
Progress: 300/701 images processed...
Progress: 400/701 images processed...
Progress: 500/701 images processed...
Progress: 600/701 images processed...
Progress: 700/701 images processed...

Images Shape: (701, 224, 224, 3)
Labels Shape: (701,)
Pixel Value Min: 0.00
Pixel Value Max: 1.00

✅ All images successfully converted!


In [7]:
# Save clean CSV dataset
df['label_numeric'] = labels

# Update image path to resized folder
df['image_path_resized'] = df['Image Name'].apply(
    lambda x: os.path.join(resized_folder, x)
)

# Save
df.to_csv('../glaucoma_clean_dataset.csv', index=False)

print("✅ Clean dataset saved!")
print(f"Total data: {len(df)}")
print(df.head())

✅ Clean dataset saved!
Total data: 701
  Image Name  Patient Label  Quality Score  label_numeric         image_path  \
0    0_0.jpg        0  GON+           6.18              1  ../Images/0_0.jpg   
1    1_0.jpg        1  GON+           5.31              1  ../Images/1_0.jpg   
2    1_1.jpg        1  GON+           4.37              1  ../Images/1_1.jpg   
3    2_1.jpg        2  GON+           4.24              1  ../Images/2_1.jpg   
4    3_0.jpg        3  GON+           5.95              1  ../Images/3_0.jpg   

          image_path_resized  
0  ../Images_resized/0_0.jpg  
1  ../Images_resized/1_0.jpg  
2  ../Images_resized/1_1.jpg  
3  ../Images_resized/2_1.jpg  
4  ../Images_resized/3_0.jpg  


PENANGANAN DATA IMBALANCE

In [8]:
import random
import shutil

# Folder for augmented images
augmented_folder = '../Images_augmented'
if not os.path.exists(augmented_folder):
    os.makedirs(augmented_folder)

# Copy all resized photos to the augmented folder first
for img_name in df['Image Name']:
    src = os.path.join(resized_folder, img_name)
    dst = os.path.join(augmented_folder, img_name)
    shutil.copy2(src, dst)

print(f"✅ All original photos copied: {len(os.listdir(augmented_folder))} photos")

# Augmentasi hanya GON-
gon_minus = df[df['Label'] == 'GON-']['Image Name'].values
print(f"\nGON- count before augmentation: {len(gon_minus)}")

# Target: tambah ~300 foto GON- baru
augmented_rows = []
aug_count = 0
target = 300

for img_name in gon_minus:
    if aug_count >= target:
        break
        
    img_path = os.path.join(resized_folder, img_name)
    img = Image.open(img_path).convert('RGB')
    
    # Augmentasi 1: Flip horizontal
    img_flip = img.transpose(Image.FLIP_LEFT_RIGHT)
    flip_name = f"aug_flip_{img_name}"
    img_flip.save(os.path.join(augmented_folder, flip_name))
    augmented_rows.append({
        'Image Name': flip_name,
        'Patient': df[df['Image Name']==img_name]['Patient'].values[0],
        'Label': 'GON-',
        'Quality Score': df[df['Image Name']==img_name]['Quality Score'].values[0],
        'label_numeric': 0,
        'image_path': os.path.join('../Images', img_name),
        'image_path_resized': os.path.join(augmented_folder, flip_name)
    })
    aug_count += 1
    
    if aug_count >= target:
        break
    
    # Augmentasi 2: Rotate 90
    img_rot = img.rotate(90)
    rot_name = f"aug_rot_{img_name}"
    img_rot.save(os.path.join(augmented_folder, rot_name))
    augmented_rows.append({
        'Image Name': rot_name,
        'Patient': df[df['Image Name']==img_name]['Patient'].values[0],
        'Label': 'GON-',
        'Quality Score': df[df['Image Name']==img_name]['Quality Score'].values[0],
        'label_numeric': 0,
        'image_path': os.path.join('../Images', img_name),
        'image_path_resized': os.path.join(augmented_folder, rot_name)
    })
    aug_count += 1

print(f"Augmented photo created: {aug_count}")
print(f"GON- count after augmentation: {len(gon_minus) + aug_count}")

✅ All original photos copied: 1001 photos

GON- count before augmentation: 192
Augmented photo created: 300
GON- count after augmentation: 492


In [9]:
# Gabungkan data augmentasi ke dataframe
df_augmented = pd.DataFrame(augmented_rows)
df_final = pd.concat([df, df_augmented], ignore_index=True)

print("Distribution after augmentation:")
print(df_final['Label'].value_counts())
print(f"\nTotal data: {len(df_final)}")

# Simpan dataset final
df_final.to_csv('../glaucoma_augmented_dataset.csv', index=False)
print("\n✅ glaucoma_augmented_dataset.csv successfully saved!")

Distribution after augmentation:
Label
GON+    509
GON-    492
Name: count, dtype: int64

Total data: 1001

✅ glaucoma_augmented_dataset.csv successfully saved!


Dataset awal imbalance 73% vs 27%, model bisa bias prediksi GON+. Kami augmentasi foto GON- dengan flip dan rotate sehingga distribusi menjadi ~50:50